Overview & Teori Dasar
Pada bab ini, dipelajari cara menangani overfitting dan mengoptimalkan proses pelatihan neural network:

Overfitting vs Generalization: Overfitting terjadi ketika model menghafal noise pada data latihan alih-alih mempelajari pola umum (signal).

Dropout: Teknik regularisasi yang mematikan neuron secara acak selama pelatihan agar node tidak saling bergantung (co-adaptation).

Batching (Mini-batch SGD): Memproses data latihan dalam kelompok kecil (batch) untuk meningkatkan kecepatan dan stabilitas konvergensi gradien.

1. Dampak Overfitting dan Penggunaan Dropout
Dropout mematikan sebagian neuron secara acak dengan mengalikan hidden layer dengan topeng acak (random mask) bernilai 0 atau 1, lalu mengalikan hasilnya dengan nilai skala agar total sinyal tetap seimbang.

In [1]:
import numpy as np

# Set seed agar hasil acak konsisten
np.random.seed(1)

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

# Data sederhana
i = np.array([1, 0, 1])
target = 1
alpha = 0.1

weights_0_1 = 2 * np.random.random((3, 4)) - 1
weights_1_2 = 2 * np.random.random((4, 1)) - 1

# Forward propagation dengan Dropout (rate = 50%)
dropout_mask = np.random.randint(2, size=(1, 4)) # Menghasilkan angka 0 atau 1 secara acak

layer_0 = i.reshape(1, 3)
layer_1 = relu(np.dot(layer_0, weights_0_1))

# Terapkan dropout mask dan kalikan 2 (1 / (1 - rate)) untuk scaling
layer_1 *= dropout_mask * 2
layer_2 = np.dot(layer_1, weights_1_2)

print("Layer 1 setelah Dropout:\n", layer_1)
print("Prediksi Layer 2:", layer_2)

Layer 1 setelah Dropout:
 [[-0.          1.03656491 -0.         -0.        ]]
Prediksi Layer 2: [[0.78388653]]


Penjelasan Teoretis:

Dropout Rate: Persentase neuron yang dimatikan (misal 50%).

Scaling (* 2): Memastikan nilai ekspektasi jumlah sinyal masukan ke layer berikutnya tetap sama baik saat pelatihan (training) maupun saat pengujian (inference).

2. Pelatihan Menggunakan Mini-Batch Gradient Descent
Mini-batch memproses sejumlah sampel data sekaligus sebelum melakukan update bobot.

In [2]:
import numpy as np

np.random.seed(1)

# Dataset dummy: 4 sampel data
streetlights = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
])
walk_vs_stop = np.array([[1, 1, 0, 0]]).T

alpha = 0.1
batch_size = 2 # Ukuran batch = 2 sampel per update bobot

weights_0_1 = 2 * np.random.random((3, 4)) - 1
weights_1_2 = 2 * np.random.random((4, 1)) - 1

for iteration in range(50):
    error = 0
    for i in range(int(len(streetlights) / batch_size)):
        # Ambil batch data
        batch_start = i * batch_size
        batch_end = (i + 1) * batch_size

        layer_0 = streetlights[batch_start:batch_end]

        # Forward pass
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)

        # Error & Delta
        error += np.sum((layer_2 - walk_vs_stop[batch_start:batch_end]) ** 2)
        layer_2_delta = (layer_2 - walk_vs_stop[batch_start:batch_end]) / batch_size

        # Backprop
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)

        # Update Bobot
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

    if iteration % 10 == 9:
        print(f"Iterasi {iteration+1:2d} | Error: {error:.6f}")

Iterasi 10 | Error: 0.953484
Iterasi 20 | Error: 0.707665
Iterasi 30 | Error: 0.574377
Iterasi 40 | Error: 0.518969
Iterasi 50 | Error: 0.470354


Penjelasan Teoretis:

Full-batch: Mengolah seluruh dataset sebelum update bobot (lambat jika data besar).

Stochastic (SGD): Update bobot per 1 sampel (cepat, namun varians gradien sangat acak).

Mini-batch: Kombinasi terbaik antara efisiensi komputasi matriks dan stabilitas pembaruan bobot.